## Design and Optimisation of Mixing Process in Stirred Tank using ANSYS fluent and Python
##### This script performs response surface analysis and optimisation on data from simulation experiments using a Central Composite Design (CCD) table. It builds and fits a second-order polynomial model to the simulation dataset, conducts ANOVA to identify significant factors and their interactions, and generates publication-quality plots including residuals analysis, 3D response surface and 2D contour plots. Finally, it performs a multi-objective genetic algorithm (MOGA) optimisation to identiy the optimum combination of the design parameters. It starts by installing the necessary python packages, libraries and dependencies such as pandas numpy seaborn matplotlib statsmodels openpyxl pyDOE3, scikit-learn etc. Start running the script by importing the necessary python packages and libraries

#### Table of Contents

1. [Loads CFD Simulation Data File and Display the DoE Table](#1-loads-cfd-simulation-data-file-and-display-the-doe-table)  
2. [Performs ANOVA and Compute Mean and Interaction Effects](#2-performs-anova-and-compute-mean-and-interaction-effects)  
3. [Creates Response Surface and Contour Plots](#3-creates-response-surface-and-contour-plots)  
4. [Performs Multi-Objective Optimisation using Polynomial Regression](#4-performs-multi-objective-optimisation-using-polynomial-regression) 

In [ ]:
# Import all the necessary python packages and libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pyDOE3
import re
import random
import itertools
import warnings
import random
import statsmodels.api as sm
import statsmodels.formula.api as smf
import sys
from itertools import combinations
from pathlib import Path
from statsmodels.formula.api import ols
from mpl_toolkits.mplot3d import Axes3D
from sklearn.linear_model import LinearRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from scipy.interpolate import RBFInterpolator
from sklearn.metrics import mean_squared_error, r2_score
warnings.filterwarnings("ignore")

## 1. Loads CFD Simulation Data File and Display the DoE Table

In [ ]:
# A script that loads Excel Data File from a Directory and display the DoE Table
# Install the required Python libraries if you haven't already:
# - Pandas and pyDoE3 (for DoE creation and analysis)

file_path = 'Mixing_tank_single_response.xlsx'  # If the file is in the current directory
data = pd.read_excel(file_path)   # Load the data from Excel

print('\n========== FULL CCD TABLE ==========')
print(f'Number of runs: {len(data)}')
print(f'Number of variables: {len(data.columns)}\n')
print(data.to_string(index=False))

## 2. Performs ANOVA and Compute Mean and Interaction Effects

In [ ]:
# A script that computes mean effects of individual factors and their pairwise interactions, perform ANOVA to identify significant effects, 
# and evaluates the goodness of fit for a linear model with main effects, interactions, and quadratic terms
# It also includes visualizations of mean effects and interactions.
# Assumptions:
# Excel file: 'Mixing_tank_1.xlsx' (in current folder) — adjust filename if necessary.
# Desired factor names (human-friendly): 'Baffle Thickness', 'Baffle Length', 'Blade Chord', 'Impeller RPM'
# Response column name is 'Average k' (kept for compatibility)

file_path = 'Mixing_tank_single_response.xlsx'  # Excel file located in the current directory
data = pd.read_excel(file_path) # Load the data from Excel

# Ensure column names have no leading/trailing whitespace and factor names match dataframe
data.columns = data.columns.str.strip()

# Define/clean factor and response names (adjust if needed)
factors = ['Baffle Thickness', 'Baffle Length', 'Blade Chord', 'Impeller RPM']   # corrected name (no trailing space)
response = 'Average k'

# Make sure the global factor_names (if present) is consistent with cleaned columns
try:
    factor_names = [fn.strip() for fn in factor_names]
except NameError:
    factor_names = factors.copy()

# Calculate mean effect of each factor
mean_effects = data[factor_names].mean()

# Plot mean effects as a bar chart
plt.figure(figsize=(8, 5))
sns.barplot(x=mean_effects.index, y=mean_effects.values, palette="viridis")
plt.title("Mean Effect of Each Factor")
plt.ylabel("Mean Value")
plt.xlabel("Factors")
plt.tight_layout()
plt.show()

# Calculate mean pairwise interaction effects for all pairs of factors
interaction_means = {}
for f1, f2 in combinations(factor_names, 2):
    interaction_term = data[f1] * data[f2]
    interaction_means[f"{f1}*{f2}"] = interaction_term.mean()

# Convert to Series for plotting
interaction_means_series = pd.Series(interaction_means)

# Plot mean interaction effects as a bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x=interaction_means_series.index, y=interaction_means_series.values, palette="mako")
plt.title("Mean Interaction Effect of Factor Pairs")
plt.ylabel("Mean Interaction Value")
plt.xlabel("Factor Pairs")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Construct the formula for the linear model including main effects, 2-factor interactions, and quadratic terms
terms = factors.copy()

# Add 2-factor interactions
for combo in combinations(factors, 2):
    terms.append(f"{combo[0]}:{combo[1]}")

# Add quadratic terms
for f in factors:
    terms.append(f"I({f}**2)")

formula = f"{response} ~ " + " + ".join(terms)

# print("\n ANOVA completed and results printed on screen.")
print(formula)

# Fit the model and compute Type II ANOVA
# Drop missing values before fitting the model
data_clean = data.dropna()

# Basic validation and cleanup
if not isinstance(formula, str):
	raise TypeError("The variable 'formula' must be a string containing a valid statsmodels formula.")
data_clean.columns = data_clean.columns.str.strip()

# Many column names contain spaces; Patsy/statsmodels formulas require valid identifiers.
# Create a safe mapping (replace spaces and some other problematic chars) and update both the dataframe and the formula.
safe_map = {col: col.strip().replace(' ', '_').replace('-', '_') for col in data_clean.columns}
data_clean = data_clean.rename(columns=safe_map)

# Update the formula to use the safe column names
safe_formula = formula
for orig, safe in safe_map.items():
	# Replace occurrences of the original column name in the formula with the safe name.
	# Use simple replace because names may appear in interactions and I(...) terms.
	safe_formula = safe_formula.replace(orig, safe)

# Fit the model and compute Type II ANOVA (appropriate for balanced designs)
try:
	model = ols(safe_formula, data=data_clean).fit()
	anova_table = sm.stats.anova_lm(model, typ=2)  # Type II ANOVA
	print("Model fitted and ANOVA table computed successfully.")
except SyntaxError:
	print("SyntaxError encountered while parsing/executing the model fit. Check that the formula is valid after replacing spaces in column names.")
	raise
except Exception as e:
	print(f"Error fitting the model or computing ANOVA: {e}")
	raise

# Print the ANOVA table
print("\n ANOVA completed and results printed on screen.")
print(anova_table.round(4))

# Identify significant factors/interactions (p < 0.05)
alpha = 0.05
significant = anova_table[anova_table['PR(>F)'] < alpha]

print("\n========== SIGNIFICANT TERMS (p < 0.05) ==========")
if not significant.empty:
    print(significant.round(4))
else:
    print("No statistically significant effects found (p >= 0.05).")

# Print Model Summary and Goodness-of-Fit Statistics
# print("\n========== MODEL SUMMARY ==========")
print(model.summary())

print("\n========== GOODNESS OF FIT METRICS ==========")
print(f"R-squared       : {model.rsquared:.4f}")
print(f"Adj. R-squared  : {model.rsquared_adj:.4f}")
print(f"F-statistic     : {model.fvalue:.4f}")
print(f"p-value (Model) : {model.f_pvalue:.4e}")

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(model.model.endog, model.fittedvalues))
print(f"RMSE            : {rmse:.4f}")

# Repeat this sequence for the remaining responses in the Excel file "Mixing_tank_multiple_responses.xlsx" located in the current directory

## 3. Creates Response Surface and Contour Plots

In [ ]:
# A Script that Creates Response Surface and Contour Plots from ANOVA Analysis using OLS regression on a dataset read from an Excel file.
# Print the model formular with Linear and Quadratic terms, and the two most significant predictors (main effects) with their p-values
# Assumptions:
# Excel file: 'Mixing_tank_1.xlsx' (in current folder) — adjust filename if necessary
# Desired factor names (human-friendly): 'Baffle Thickness', 'Baffle Length', 'Blade Chord', 'Impeller RPM'
# Response column name is 'Average k' (kept for compatibility)

xlsx_file = "Mixing_tank_single_response.xlsx"
sheet_name = 0  # 0 = first sheet, or use sheet name string

# Optional: set explicitly. If None, script auto-selects numeric columns.
response_var = None            # e.g., "Average k"
predictor_vars = None          # e.g., ["Baffle Thickness", "Baffle Length", "Blade Chord", "Impeller RPM"]

n_grid = 40

def make_safe_names(columns):
    """Create valid Python/statsmodels variable names and keep uniqueness."""
    used = set()
    mapping = {}
    for col in columns:
        safe = re.sub(r"\W|^(?=\d)", "_", str(col))
        if not safe:
            safe = "col"
        base = safe
        k = 1
        while safe in used:
            safe = f"{base}_{k}"
            k += 1
        used.add(safe)
        mapping[col] = safe
    return mapping

def load_excel_checked(file_name, sheet):
    """Load Excel file with clear error messages."""
    path = Path(file_name)

    if not path.exists():
        raise FileNotFoundError(
            f"Excel file not found: '{path}'.\n"
            f"Tip: Use an absolute path or place the file in the current working directory:\n"
            f"  {Path.cwd()}"
        )

    try:
        return pd.read_excel(path, sheet_name=sheet)
    except ValueError as e:
        # Common for invalid sheet name/index
        raise ValueError(
            f"Could not read sheet '{sheet}' from '{path.name}'. "
            f"Check that the sheet name/index is correct."
        ) from e
    except Exception as e:
        raise RuntimeError(f"Failed to read Excel file '{path}': {e}") from e


def validate_columns(data_raw, response, predictors):
    """Validate configured response/predictor names and choose defaults."""
    all_cols = data_raw.columns.tolist()
    numeric_cols = data_raw.select_dtypes(include=[np.number]).columns.tolist()

    if len(numeric_cols) < 3:
        raise ValueError(
            "Need at least 3 numeric columns in the dataset "
            "(1 response + at least 2 predictors)."
        )

    # Response selection
    if response is None:
        local_response = numeric_cols[-1]
    else:
        if response not in all_cols:
            raise ValueError(
                f"Configured response_var '{response}' was not found.\n"
                f"Available columns: {all_cols}"
            )
        local_response = response

    # Predictor selection
    if predictors is None:
        local_predictors = [c for c in numeric_cols if c != local_response][:4]
    else:
        if not isinstance(predictors, (list, tuple)) or len(predictors) == 0:
            raise ValueError("predictor_vars must be a non-empty list/tuple of column names.")
        missing = [c for c in predictors if c not in all_cols]
        if missing:
            raise ValueError(
                f"Configured predictor column(s) not found: {missing}\n"
                f"Available columns: {all_cols}"
            )
        local_predictors = list(predictors)

    # Basic predictor checks
    local_predictors = [c for c in local_predictors if c != local_response]
    if len(local_predictors) < 2:
        raise ValueError("Need at least 2 predictors after validation.")

    # Numeric checks for modeling columns
    required_cols = [local_response] + local_predictors
    non_numeric = [c for c in required_cols if c not in numeric_cols]
    if non_numeric:
        raise ValueError(
            f"These required columns are not numeric: {non_numeric}. "
            f"Please choose numeric columns for response and predictors."
        )

    return local_response, local_predictors

def main():
    if not isinstance(n_grid, int) or n_grid < 2:
        raise ValueError("n_grid must be an integer >= 2.")

    # Load data with friendly path/sheet errors
    data_raw = load_excel_checked(xlsx_file, sheet_name)

    # Validate and choose columns
    local_response, local_predictors = validate_columns(data_raw, response_var, predictor_vars)

    # Keep required columns, drop missing rows
    required_cols = [local_response] + local_predictors
    data = data_raw[required_cols].dropna().copy()
    if data.empty:
        raise ValueError(
            "No data left after dropping rows with missing values in required columns: "
            f"{required_cols}"
        )

    # Rename to safe names for formula compatibility
    name_map = make_safe_names(data.columns)
    inv_map = {v: k for k, v in name_map.items()}
    data = data.rename(columns=name_map)

    response_safe = name_map[local_response]
    predictors_safe = [name_map[c] for c in local_predictors]

    # Build polynomial model:
    # response ~ main effects + pairwise interactions + squared terms
    main_terms = " + ".join(predictors_safe)
    interaction_terms = " + ".join(
        f"{a}:{b}" for i, a in enumerate(predictors_safe) for b in predictors_safe[i + 1:]
    )
    quadratic_terms = " + ".join(f"I({v}**2)" for v in predictors_safe)

    formula = f"{response_safe} ~ {main_terms}"
    if interaction_terms:
        formula += " + " + interaction_terms
    if quadratic_terms:
        formula += " + " + quadratic_terms

    mdl = smf.ols(formula=formula, data=data).fit()

    # Identify two most significant main-effect predictors
    pvals = mdl.pvalues
    main_effect_pvals = {
        v: pvals[v] for v in predictors_safe
        if v in pvals.index and pd.notna(pvals[v])
    }
    if len(main_effect_pvals) < 2:
        raise ValueError(
            "Could not find at least 2 valid main-effect predictors in the fitted model."
        )

    sig_predictors = sorted(main_effect_pvals, key=main_effect_pvals.get)[:2]
    p1, p2 = sig_predictors[0], sig_predictors[1]

    # Grid for the two significant predictors
    x1 = np.linspace(data[p1].min(), data[p1].max(), n_grid)
    x2 = np.linspace(data[p2].min(), data[p2].max(), n_grid)
    X1, X2 = np.meshgrid(x1, x2)

    # Prediction table: vary p1/p2, hold others at mean
    n_rows = X1.size
    predict_df = pd.DataFrame(index=np.arange(n_rows), columns=predictors_safe, dtype=float)

    for v in predictors_safe:
        if v == p1:
            predict_df[v] = X1.ravel()
        elif v == p2:
            predict_df[v] = X2.ravel()
        else:
            predict_df[v] = data[v].mean()

    y_hat = mdl.predict(predict_df).to_numpy()
    Yhat_grid = y_hat.reshape(n_grid, n_grid)

    # Plot: 3D surface + contour
    fig = plt.figure(figsize=(14, 6))
    fig.suptitle("3D Surface Plot and 2D Contour Plot", fontsize=14)

    # 3D surface
    ax1 = fig.add_subplot(1, 2, 1, projection="3d")
    surf = ax1.plot_surface(X1, X2, Yhat_grid, cmap="viridis", edgecolor="none")
    ax1.set_xlabel(str(inv_map[p1]))
    ax1.set_ylabel(str(inv_map[p2]))
    ax1.set_zlabel(str(local_response))
    ax1.set_title("3D Surface Plot")
    ax1.view_init(elev=30, azim=135)
    fig.colorbar(surf, ax=ax1, shrink=0.6)

    # Contour
    ax2 = fig.add_subplot(1, 2, 2)
    c = ax2.contourf(X1, X2, Yhat_grid, levels=20, cmap="viridis")
    ax2.set_xlabel(str(inv_map[p1]))
    ax2.set_ylabel(str(inv_map[p2]))
    ax2.set_title("2D Contour Plot")
    fig.colorbar(c, ax=ax2)

    plt.tight_layout()
    plt.show()

    print("\nModel formula:")
    print(formula)
    print("\nMost significant predictors (main effects):")
    print(f"1) {inv_map[p1]} (p = {main_effect_pvals[p1]:.4g})")
    print(f"2) {inv_map[p2]} (p = {main_effect_pvals[p2]:.4g})")

if __name__ == "__main__":
    try:
        main()
    except Exception as err:
        print(f"\nError: {err}")
        sys.exit(1)

## 4. Performs Multi-Objective Optimisation using Polynomial Regression

In [ ]:
# =============================================================================
#  A Script that Performs Multi-Objective Genetic Algorithm (NSGA-II-like) using Second Degree (Quadratic) Polynomial Regression
#  Excel file loaded from the same directory as this script
#  Excel file: first 4 cols = factors, last 4 = responses
#  Surrogate model: (Polynomial Regression)
#  Objective: minimize Y3, while treating Y1 and Y4 as functional constraints subject to Y1 >= 0.03 and Y4 >= 2
# =============================================================================

# Load necessary python libraries
import numpy as np
import pandas as pd
import re
import random
from pathlib import Path
from sklearn.linear_model import LinearRegression

# Sanitise column names to valid Python identifiers
def make_valid_name(name: str) -> str:
    s = name.strip().replace(" ", "_")
    s = re.sub(r"[^0-9a-zA-Z_]", "_", s)
    if re.match(r"^\d", s):
        s = f"x_{s}"
    return s

# Fast non-dominated sorting (NSGA-II front construction)
def make_rsm_features(vars_):
    v = np.asarray(vars_, dtype=float).ravel()
    n = v.size
    linear = v
    quad = v ** 2
    inter = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            inter.append(v[i] * v[j])
    return np.concatenate([linear, quad, np.array(inter, dtype=float)]).reshape(1, -1)


def build_rsm_matrix(X):
    X = np.asarray(X, dtype=float)
    n = X.shape[1]
    linear = X
    quad = X ** 2
    inter_cols = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            inter_cols.append((X[:, i] * X[:, j]).reshape(-1, 1))
    inter = np.hstack(inter_cols) if inter_cols else np.empty((X.shape[0], 0))
    return np.hstack([linear, quad, inter])


def dominates(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    return np.all(a <= b) and np.any(a < b)


def fast_non_dominated_sort(objs):
    N = objs.shape[0]
    S = [[] for _ in range(N)]
    n = np.zeros(N, dtype=int)
    fronts = []
    F = []

    for p in range(N):
        for q in range(N):
            if dominates(objs[p], objs[q]):
                S[p].append(q)
            elif dominates(objs[q], objs[p]):
                n[p] += 1
        if n[p] == 0:
            F.append(p)

    fronts.append(F)
    i = 0
    while len(fronts[i]) > 0:
        Q = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    Q.append(q)
        i += 1
        fronts.append(Q)

    if len(fronts[-1]) == 0:
        fronts.pop()
    return fronts


def crowding_distance(front, objs):
    l = len(front)
    if l == 0:
        return np.array([])
    dist = np.zeros(l, dtype=float)
    m = objs.shape[1]

    for k in range(m):
        vals = objs[front, k]
        idx = np.argsort(vals)
        sorted_vals = vals[idx]
        dist[idx[0]] = np.inf
        dist[idx[-1]] = np.inf
        fmin, fmax = sorted_vals[0], sorted_vals[-1]
        if fmax == fmin:
            continue
        for j in range(1, l - 1):
            dist[idx[j]] += (sorted_vals[j + 1] - sorted_vals[j - 1]) / (fmax - fmin)
    return dist


def tournament_selection(pop, objs):
    i1 = random.randint(0, len(pop) - 1)
    i2 = random.randint(0, len(pop) - 1)
    if dominates(objs[i1], objs[i2]):
        return pop[i1].copy()
    elif dominates(objs[i2], objs[i1]):
        return pop[i2].copy()
    else:
        lo, hi = min(i1, i2), max(i1, i2)
        return pop[random.randint(lo, hi)].copy()


# Simulated Binary Crossover (SBX)
def sbx_crossover(p1, p2, low, up, eta_c):
    c1 = p1.copy()
    c2 = p2.copy()
    for i in range(p1.size):
        if random.random() <= 0.5:
            if abs(p1[i] - p2[i]) > 1e-14:
                x1, x2 = min(p1[i], p2[i]), max(p1[i], p2[i])
                rand_beta = random.random()

                beta = 1 + 2 * (x1 - low[i]) / (x2 - x1)
                alpha = 2 - beta ** (-(eta_c + 1))
                if rand_beta <= 1 / alpha:
                    betaq = (rand_beta * alpha) ** (1 / (eta_c + 1))
                else:
                    betaq = (1 / (2 - rand_beta * alpha)) ** (1 / (eta_c + 1))
                c1[i] = 0.5 * ((x1 + x2) - betaq * (x2 - x1))

                beta = 1 + 2 * (up[i] - x2) / (x2 - x1)
                alpha = 2 - beta ** (-(eta_c + 1))
                if rand_beta <= 1 / alpha:
                    betaq = (rand_beta * alpha) ** (1 / (eta_c + 1))
                else:
                    betaq = (1 / (2 - rand_beta * alpha)) ** (1 / (eta_c + 1))
                c2[i] = 0.5 * ((x1 + x2) + betaq * (x2 - x1))

                c1[i] = min(max(c1[i], low[i]), up[i])
                c2[i] = min(max(c2[i], low[i]), up[i])
    return c1, c2

# Polynomial mutation
def polynomial_mutation(child, low, up, eta_m, mutprob):
    out = child.copy()
    for i in range(out.size):
        if random.random() < mutprob:
            x = out[i]
            delta1 = (x - low[i]) / (up[i] - low[i])
            delta2 = (up[i] - x) / (up[i] - low[i])
            rand_mut = random.random()
            mut_pow = 1 / (eta_m + 1)

            if rand_mut < 0.5:
                xy = 1 - delta1
                val = 2 * rand_mut + (1 - 2 * rand_mut) * (xy ** (eta_m + 1))
                deltaq = val ** mut_pow - 1
            else:
                xy = 1 - delta2
                val = 2 * (1 - rand_mut) + 2 * (rand_mut - 0.5) * (xy ** (eta_m + 1))
                deltaq = 1 - val ** mut_pow

            x = x + deltaq * (up[i] - low[i])
            out[i] = min(max(x, low[i]), up[i])
    return out


# Evaluate a candidate solution using trained surrogate models
# Obj1 = predicted Y3 (minimize)
# Obj2 = total constraint violation for Y1 and Y4 (minimize)

def evaluate_solution(x, models, y1_col, y3_col, y4_col, y1_lb, y4_lb):
    feat = make_rsm_features(x)
    preds = {name: float(model.predict(feat)[0]) for name, model in models.items()}
    y3 = preds[y3_col]
    viol = max(0.0, y1_lb - preds[y1_col]) + max(0.0, y4_lb - preds[y4_col])
    return np.array([y3, viol], dtype=float), preds


# Get script directory safely (works in script + notebook contexts)
def get_base_dir() -> Path:
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


def main():
    # Reproducibility
    np.random.seed(42)
    random.seed(42)

    # Input file (script dir if .py, cwd if notebook)
    script_dir = get_base_dir()
    file_path = script_dir / "Mixing_tank_multiple_responses.xlsx"

    # Read and clean data
    data = pd.read_excel(file_path)
    data.columns = [make_valid_name(c) for c in data.columns]
    data = data.dropna().reset_index(drop=True)

    predictors = list(data.columns[:4])
    responses = list(data.columns[4:8])
    y1_col = responses[0]
    y3_col = responses[2]
    y4_col = responses[3]

    X = data[predictors].to_numpy(dtype=float)
    X_rsm = build_rsm_matrix(X)

    # Fit one Surrogate model per response and compute training metrics
    models = {}
    metrics = {}
    for r in responses:
        y = data[r].to_numpy(dtype=float)
        mdl = LinearRegression(fit_intercept=True)
        mdl.fit(X_rsm, y)
        y_pred = mdl.predict(X_rsm)
        mse = float(np.mean((y - y_pred) ** 2))
        r2 = float(1 - np.sum((y - y_pred) ** 2) / np.sum((y - np.mean(y)) ** 2))
        models[r] = mdl
        metrics[r] = {"MSE": mse, "R2": r2}

    metrics_flat = pd.DataFrame({
        "Response": list(metrics.keys()),
        "R2": [metrics[k]["R2"] for k in metrics.keys()],
        "MSE": [metrics[k]["MSE"] for k in metrics.keys()],
    })
    print("Surrogate model metrics (R2 and MSE):")
    print(metrics_flat)

    # GA parameters
    POP_SIZE = 100
    NGEN = 80
    CXPB = 0.9
    MUTPB = 0.2
    ETA_C = 20.0
    ETA_M = 20.0

    bounds = np.vstack([X.min(axis=0), X.max(axis=0)]).T
    low = bounds[:, 0]
    up = bounds[:, 1]

    y1_lb = 0.03
    y4_lb = 2.0

    # Initialize population
    pop = []
    objs = np.zeros((POP_SIZE, 2), dtype=float)
    preds = []
    for i in range(POP_SIZE):
        ind = low + (up - low) * np.random.rand(low.size)
        pop.append(ind)
        objs[i], p = evaluate_solution(ind, models, y1_col, y3_col, y4_col, y1_lb, y4_lb)
        preds.append(p)

    # Evolution loop
    for _ in range(NGEN):
        offspring = []
        while len(offspring) < POP_SIZE:
            p1 = tournament_selection(pop, objs)
            p2 = tournament_selection(pop, objs)

            if random.random() < CXPB:
                c1, c2 = sbx_crossover(p1, p2, low, up, ETA_C)
            else:
                c1, c2 = p1.copy(), p2.copy()

            c1 = polynomial_mutation(c1, low, up, ETA_M, MUTPB)
            c2 = polynomial_mutation(c2, low, up, ETA_M, MUTPB)
            offspring.append(c1)
            if len(offspring) < POP_SIZE:
                offspring.append(c2)

        off_objs = np.zeros((POP_SIZE, 2), dtype=float)
        off_preds = []
        for i in range(POP_SIZE):
            off_objs[i], p = evaluate_solution(offspring[i], models, y1_col, y3_col, y4_col, y1_lb, y4_lb)
            off_preds.append(p)

        combined = pop + offspring
        combined_objs = np.vstack([objs, off_objs])
        combined_preds = preds + off_preds

        fronts = fast_non_dominated_sort(combined_objs)

        new_pop, new_objs, new_preds = [], [], []
        for front in fronts:
            if len(new_pop) + len(front) > POP_SIZE:
                dist = crowding_distance(front, combined_objs)
                idx_sorted = np.argsort(-dist)
                for k in idx_sorted:
                    idx = front[k]
                    if len(new_pop) < POP_SIZE:
                        new_pop.append(combined[idx])
                        new_objs.append(combined_objs[idx])
                        new_preds.append(combined_preds[idx])
                    else:
                        break
                break
            else:
                for idx in front:
                    new_pop.append(combined[idx])
                    new_objs.append(combined_objs[idx])
                    new_preds.append(combined_preds[idx])

        pop = new_pop
        objs = np.array(new_objs, dtype=float)
        preds = new_preds

    # Pareto front
    fronts = fast_non_dominated_sort(objs)
    pf_idx = fronts[0]

    rows = []
    for i in pf_idx:
        row = {predictors[j]: pop[i][j] for j in range(len(predictors))}
        row["Y3_obj"] = objs[i, 0]
        row["total_violation"] = objs[i, 1]
        row.update(preds[i])
        rows.append(row)

    pareto_tbl = pd.DataFrame(rows)
    feasible = pareto_tbl[pareto_tbl["total_violation"] <= 1e-9].copy()

    if feasible.empty:
        feasible = pareto_tbl.nsmallest(min(10, len(pareto_tbl)), "total_violation").copy()

    best = feasible.nsmallest(1, "Y3_obj").copy()

    # Save outputs to Excel (same directory as this script)
    out_file = script_dir / "MOGA_Poly_Optimisation_results.xlsx"

    with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
        pareto_tbl.to_excel(writer, sheet_name="Pareto_front", index=False)
        feasible.to_excel(writer, sheet_name="Feasible", index=False)
        metrics_flat.to_excel(writer, sheet_name="Model_Metrics", index=False)

    # Print surrogate model statistics and the feasible solution
    print("\nBest feasible solution:")
    cols = predictors + responses + ["total_violation"]
    cols = [c for c in cols if c in best.columns]
    print(best[cols])
    print(f"\nResults saved to {out_file}")


if __name__ == "__main__":
    main()